<a href="https://colab.research.google.com/github/jjean2006/EDA-Experiments/blob/main/Experiment5_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from scipy.stats import zscore

train_df = pd.read_csv('/content/fraudTrain.csv')
test_df = pd.read_csv('/content/fraudTest.csv')

numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
if 'is_fraud' in numeric_cols: numeric_cols.remove('is_fraud')
if 'Unnamed: 0' in numeric_cols: numeric_cols.remove('Unnamed: 0')

print(f'Numerical columns used for outlier detection: {numeric_cols}')
display(train_df[numeric_cols].head())

Numerical columns used for outlier detection: ['cc_num', 'amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long']


,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long
0,2703186189652095,4.97,28654.0,36.0788,-81.1781,3495.0,1.325376e+09,36.011293,-82.048315
1,630423337322,107.23,99160.0,48.8878,-118.2105,149.0,1.325376e+09,49.159047,-118.186462
2,38859492057661,220.11,83252.0,42.1808,-112.2620,4154.0,1.325376e+09,43.150704,-112.154481
3,3534093764340240,45.00,59632.0,46.2306,-112.1138,1939.0,1.325376e+09,47.034331,-112.561071
4,375534208663984,41.96,24433.0,38.4207,-79.4629,99.0,1.325376e+09,38.674999,-78.632459


In [16]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)

train_df['iso_outlier'] = iso_forest.fit_predict(train_df[numeric_cols])
test_df['iso_outlier'] = iso_forest.predict(test_df[numeric_cols])

print('Isolation Forest Outliers detected (Train):', (train_df['iso_outlier'] == -1).sum())
print('Isolation Forest Outliers detected (Test):', (test_df['iso_outlier'] == -1).sum())

Isolation Forest Outliers detected (Train): 34103
Isolation Forest Outliers detected (Test): 51854


In [17]:
z_scores = np.abs(zscore(train_df[numeric_cols]))
train_df['zscore_outlier'] = (z_scores > 3).any(axis=1)

test_z_scores = np.abs(zscore(test_df[numeric_cols]))
test_df['zscore_outlier'] = (test_z_scores > 3).any(axis=1)

print('Z-Score Outliers detected (Train):', train_df['zscore_outlier'].sum())
print('Z-Score Outliers detected (Test):', test_df['zscore_outlier'].sum())

Z-Score Outliers detected (Train): 44660
Z-Score Outliers detected (Test): 55858


In [18]:
train_outlier_indices = pd.Series([False] * len(train_df))
for col in numeric_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    train_outlier_indices = train_outlier_indices | (train_df[col] < lower_bound) | (train_df[col] > upper_bound)
train_df['iqr_outlier'] = train_outlier_indices

test_outlier_indices = pd.Series([False] * len(test_df))
for col in numeric_cols:
    Q1 = test_df[col].quantile(0.25)
    Q3 = test_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    test_outlier_indices = test_outlier_indices | (test_df[col] < lower_bound) | (test_df[col] > upper_bound)
test_df['iqr_outlier'] = test_outlier_indices

print("Tukey's IQR Outliers detected (Train):", train_df['iqr_outlier'].sum())
print("Tukey's IQR Outliers detected (Test):", test_df['iqr_outlier'].sum())

Tukey's IQR Outliers detected (Train): 218453
Tukey's IQR Outliers detected (Test): 177491
